# Avaliações de interesse

- expandir uma caminhada a partir de um evento
- verificar se os pares de eventos de arestas dentro da caminhada violam as imposições do grafo base
- fazer isto com as abordagens zero-shot

In [1]:
# Dados
TEST_EDGES = "../data/labels/small_directed_mad3.5/link_prediction_edges.json"
GRAPH_PATH = "../data/processed/directed_mad3.5_ai_news_event_graph.pkl"
USE_JSON_STRS = False

In [2]:
# Modelo
SUFFIX = "ckpt-760"
# CHECKPOINT_DIR = f"/exp_local/kenzosaki/models/ai_events_gemma_3_4b_clm_r128_lr1e-4_no_json/checkpoint-{SUFFIX.split('-')[1]}"
# CHECKPOINT_DIR = f"/exp_local/kenzosaki/models/ai_events_gemma_3_4b_clm_r128_lr1e-4_no_json/best_merged"
CHECKPOINT_DIR = "/exp_local/kenzosaki/models/ai_events_gemma_3_4b_dpo/checkpoint-200"
# CHECKPOINT_DIR = f"/exp_local/kenzosaki/models/ai_events_gemma_3_4b_clm_r64_lr1e-4/checkpoint-{SUFFIX.split('-')[1]}"
DEVICE = "cuda"
MAX_SEQ_LENGTH = 1024 
EVAL_BS = 2

# Carregando modelo

In [3]:
from glm_based_event_analysis.generation.pre_trained import PreTrainedEventGenerator

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 05-31 19:36:21 [__init__.py:235] Automatically detected platform cuda.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
event_generator = PreTrainedEventGenerator.from_unsloth_ckpt(CHECKPOINT_DIR)

==((====))==  Unsloth 2026.5.2: Fast Gemma3 patching. Transformers: 4.57.2. vLLM: 0.10.0+cu126.
   \\   /|    NVIDIA RTX A5000. Num GPUs = 1. Max memory: 23.679 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 8.6. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.31. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

# Carregando dados de entrada

In [5]:
import pickle
import json
import random
from glm_based_event_analysis.utils.graph import get_node_metadata, remove_edges_from_graph

In [6]:
with open(GRAPH_PATH, "rb") as f:
    G = pickle.load(f)

with open(TEST_EDGES, "r") as f:
    test_edges = json.load(f)

In [7]:
G_train = remove_edges_from_graph(G, test_edges)
G_train_inverted = G_train.reverse()
G_inverted = G.reverse()

# Expandindo a partir de um evento

In [8]:
from glm_based_event_analysis.generation.formatting import prepare_json_str, prepare_simple_str

In [9]:
train_nodes = list(G_train.nodes())

In [10]:
u_test = random.choice(train_nodes)
u_test

'6721_google_play_pc_2026-03-11'

In [11]:
if USE_JSON_STRS:
    input_str = prepare_json_str([u_test], G)
else:
    input_str = prepare_simple_str([u_test], G)
print(input_str)

<event>
When: 2026-03-11-07:00:00
What: Google Play Games for PC is expanding its library with more premium titles and introducing cross-buy functionality with Android.
Who: Google, Android users, PC gamers
Why: To enhance the gaming experience on PC and bridge the gap between Android and PC gaming platforms.
Where: United States
How: Through software updates and integration of cross-buy features within the Google Play Games for PC application.
ID: 6721_google_play_pc_2026-03-11
</event>



In [12]:
# TODO: varias. salvar, verificar se ja existem caminhadas geradas e analisar
generated_walks = event_generator.batch_generation(
    prompts=[input_str],
    bs=EVAL_BS,
    device=DEVICE,
    max_length=MAX_SEQ_LENGTH,
    do_sample=True,
    top_p=0.9,
    #top_k=15,
    temperature=1.2,
    min_new_tokens=20
)

- Tokenizing prompts:   0%|          | 0/1 [00:00<?, ? examples/s]

In [13]:
generated_walks

[[{'event_id': '7152_google_play_pc_library_2026-03-12',
   'what': 'Google Play Games for PC is expanding its library with a Google Play Pass and introducing cross-buy functionality.',
   'who': 'Google, Google Play Pass users, PC gamers',
   'why': 'To enhance the gaming experience on PC and bridge the gap between Android and PC gaming platforms.',
   'where': 'United States',
   'how': 'Through software updates and integration of cross-buy features within the Google Play Games for PC application.'},
  {'event_id': '7169_google_play_pc_library_expansion_2026-03-16',
   'what': 'Google Play for PC is expanding its library with cross-buy games and a Google Play Pass.',
   'who': 'Google, PC gamers, Google Play Pass subscribers',
   'why': 'To expand the Google Play for PC ecosystem and cross-promote games.',
   'where': 'United States',
   'how': 'Through the expansion of the PC game library and the introduction of a Google Play Pass.'},
  {'event_id': 'PC_LIBRARY_EXPANSION_2026-03-22'

# Repetindo a geração, fornecendo mais contexto

In [14]:
from glm_based_event_analysis.random_walks.sampler import RandomWalkSampler

In [15]:
# repetindo a geração, fornecendo mais contexto
walk_size = 3 # 4 contando o nó de origem
num_walks_per_node = 3
sampler = RandomWalkSampler("none", walk_size, num_walks_per_node, inverted=True)

In [16]:
u_test = random.choice(train_nodes)
#u_test = "1754_quantum_intelligence_threat_2026-03-30"
prior_events = sampler.sample_random_walk(u_test, G_inverted)
prior_events

[np.str_('10394_pentagon_anthropic_trump_2026-03-20'),
 np.str_('10582_pentagon_anthropic_security_2026-03-24'),
 '10551_anthropic_pentagon_san_francisco_2026-03-24']

In [17]:
if USE_JSON_STRS:
    event_history_str = prepare_json_str(prior_events, G)
else:
    event_history_str = prepare_simple_str(prior_events, G)
print(event_history_str)

<event>
When: 2026-03-20-07:00:00
What: A new court filing revealed that the Pentagon informed Anthropic that their efforts were nearing alignment, shortly after Trump declared the relationship terminated.
Who: Pentagon, Anthropic, Donald Trump
Why: To resolve disagreements and potentially resume collaboration between the Pentagon and Anthropic.
Where: United States
How: Through a court filing and subsequent public statements.
ID: 10394_pentagon_anthropic_trump_2026-03-20
</event>

<event>
When: 2026-03-24-07:00:00
What: A judge suggests the Pentagon's actions towards Anthropic were motivated by punishment rather than national security concerns.
Who: Judge, Pentagon, Anthropic
Why: Alleged motivation by the Pentagon to punish Anthropic.
Where: United States
How: Through legal proceedings and a judge's assessment of the Pentagon's actions.
ID: 10582_pentagon_anthropic_security_2026-03-24
</event>

<event>
When: 2026-03-24-07:00:00
What: Anthropic is challenging a US Pentagon ban in cour

In [18]:
# TODO: varias. salvar, verificar se ja existem caminhadas geradas e analisar
generated_walks = event_generator.batch_generation(
    prompts=[event_history_str],
    bs=EVAL_BS,
    device=DEVICE,
    max_length=MAX_SEQ_LENGTH,
    do_sample=True,
    top_p=0.9,
    #top_k=15,
    temperature=1.2,
    min_new_tokens=20
)

- Tokenizing prompts:   0%|          | 0/1 [00:00<?, ? examples/s]

In [19]:
generated_walks

[[{'when': '2026-03-27-07:00:00',
   'what': 'Anthropic is challenging a Pentagon ban regarding AI court filings.',
   'who': 'Anthropic, Pentagon, AI court filings',
   'why': 'To challenge a ban on AI court filings.',
   'where': 'United States',
   'how': 'Through a legal challenge in court.',
   'event_id': 'antropic_pentagon_ban_2026-03-27'},
  {'when': '2026-03-29-07:00:00',
   'what': 'Anthropic is suing the Pentagon over AI court filings and a ban.',
   'who': 'Anthropic, Pentagon',
   'why': 'AI court filings and a ban',
   'where': 'United States',
   'how': 'Suing',
   'event_id': '10755_ai_court_ban_2026-03-29'},
  {'when': '2026-03-30-07:00:00',
   'what': 'Anthropic is suing the Pentagon regarding a ban on AI tools, with a court filing expected in early 2026.',
   'who': 'Anthropic, Pentagon',
   'why': 'To challenge a ban on AI tools.',
   'where': 'United States',
   'how': 'Through a court filing.',
   'event_id': '10886_ai_court_filing_2026-03-30'},
  {'when': '2026-0

In [20]:
# fornecer um contexto (historico) aumenta muito a qualidade da geração.

In [21]:
# testando se o modelo ainda é capaz de seguir instruções
# generated_walks = event_generator.batch_generation(
#     prompts=["what is the capital of brasil"],
#     bs=EVAL_BS,
#     device=DEVICE,
#     max_length=MAX_SEQ_LENGTH,
#     do_sample=True,
#     top_p=0.95,
#     #top_k=15,
#     temperature=1.3
# )

# generated_walks[0]